Se importan en local todos los módulos y paquetes necesarios

In [1]:
import sys
print(sys.executable)

c:\Users\sandy\Documents\GitHub\tc-sql-temuzon\venv\Scripts\python.exe


In [16]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

print("ENV cargado")
print(os.getenv("GOOGLE_APPLICATION_CREDENTIALS"))

ENV cargado
C:\Users\sandy\Documents\GitHub\tc-sql-temuzon\credentials\service-account.json


In [ ]:
import google.cloud.bigquery as bq

print("Todo OK")

Todo OK


In [19]:
from google.cloud import bigquery

client = bigquery.Client()

print("Conectado a:", client.project)

Conectado a: ds-temuzon


In [22]:
import os

dataset_id = os.getenv("BQ_DATASET_ID")

tables = client.list_tables(dataset_id)

print("Tablas:")
for table in tables:
    print(table.table_id)

Tablas:
categoria_productos
clientes
linea_pedidos
pagos
paises
pedidos
productos
resenas


# Queries de verificación

**1. Clientes por país**

In [25]:
project = os.getenv("GCP_PROJECT_ID")
dataset = os.getenv("BQ_DATASET_ID")

In [52]:
query = f"""
SELECT
  p.nombre AS pais,
  COUNT(c.id_cliente) AS total_clientes
FROM `{project}.{dataset}.clientes` c
LEFT JOIN `{project}.{dataset}.paises` p
  ON c.pais = p.id_pais
GROUP BY p.nombre
ORDER BY total_clientes DESC
"""

df1 = client.query(query).to_dataframe()
df1

c:\Users\sandy\Documents\GitHub\tc-sql-temuzon\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,pais,total_clientes
0,Vietman,64
1,Guatemala,59
2,Burkina Faso,56
3,Venezuela,56
4,Brunei Darussalam,54
5,Singapur,47
6,Estonia,44
7,Chile,43
8,Arabia Saudita,42
9,Finlandia,35


El país con más clientes al que prestamos servicio es Vietnam

**2. Productos más vendidos**

In [43]:
query = f"""
SELECT
  cp.nombre AS categoria,
  SUM(lp.cantidad) AS total_vendido
FROM `{project}.{dataset}.linea_pedidos` lp
JOIN `{project}.{dataset}.productos` pr
  ON lp.id_producto = pr.id_producto
LEFT JOIN `{project}.{dataset}.categoria_productos` cp
  ON pr.id_categoria = cp.id_categoria
GROUP BY cp.nombre
ORDER BY total_vendido DESC
LIMIT 10
"""
df2 = client.query(query).to_dataframe()
df2

c:\Users\sandy\Documents\GitHub\tc-sql-temuzon\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,categoria,total_vendido
0,wearables,1506
1,audio,1465
2,perifericos,1162
3,smartphones,933
4,laptops,911


El tipo de producto más vendido es wearables.

**3. Tiempo medio de envío**

In [31]:
query = f"""
SELECT
  AVG(DATE_DIFF(fecha_de_reparto, fecha_de_envio, DAY)) AS tiempo_medio_envio_dias
FROM `{project}.{dataset}.pedidos`
WHERE fecha_de_envio IS NOT NULL
  AND fecha_de_reparto IS NOT NULL
"""

df3 = client.query(query).to_dataframe()
df3

c:\Users\sandy\Documents\GitHub\tc-sql-temuzon\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,tiempo_medio_envio_dias
0,2.5055


El tiempo medio de días para la entrega del pedido es de 2 días y medio.

**4. Ingresos por mes**

In [37]:
query = f"""
SELECT
  FORMAT_DATE('%Y-%m', DATE(fecha_de_cobro)) AS mes,
  SUM(cantidad) AS ingresos_totales
FROM `{project}.{dataset}.pagos`
WHERE estado = 'completado'
GROUP BY mes
ORDER BY mes
"""
df4 = client.query(query).to_dataframe()
df4

c:\Users\sandy\Documents\GitHub\tc-sql-temuzon\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,mes,ingresos_totales
0,2025-05,187992.16
1,2025-06,195375.83
2,2025-07,196358.03
3,2025-08,188321.48
4,2025-09,174341.27
5,2025-10,199411.99
6,2025-11,174371.02
7,2025-12,220920.09
8,2026-01,207171.16
9,2026-02,191555.57


El mes en el que más ingresos se obtuvo fue diciembre de 2025.

**5. Valoración media de reseñas** 

In [38]:
query = f"""
SELECT
  AVG(valoracion) AS valoracion_media
FROM `{project}.{dataset}.resenas`
"""

df5 = client.query(query).to_dataframe()
df5

c:\Users\sandy\Documents\GitHub\tc-sql-temuzon\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,valoracion_media
0,3.093256


La media de reseñas sobre 5 es de 3.09.

**6. Principal canal de adquisición de portátiles (laptops)**

In [51]:
query = f"""
SELECT
  c.canal_de_adquisicion,
  COUNT(*) AS total
FROM `{project}.{dataset}.clientes` c
JOIN `{project}.{dataset}.linea_pedidos` lp
  ON c.id_cliente = lp.id_pedido
JOIN `{project}.{dataset}.productos` pr
  ON lp.id_producto = pr.id_producto
JOIN `{project}.{dataset}.categoria_productos` cp
  ON pr.id_categoria = cp.id_categoria
WHERE cp.nombre = 'laptops'
GROUP BY c.canal_de_adquisicion
ORDER BY total DESC
LIMIT 3
"""
df6 = client.query(query).to_dataframe()
df6

c:\Users\sandy\Documents\GitHub\tc-sql-temuzon\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,canal_de_adquisicion,total
0,directo,43
1,redes_sociales,40
2,publicidad_de_pago,26


La forma de adquirir portátiles más habitual es la compra directa.

**7. Producto más comprado por país**

In [50]:
query = f"""
SELECT
  pa.nombre AS pais,
  cp.nombre AS categoria,
  SUM(lp.cantidad) AS total_comprado
FROM `{project}.{dataset}.linea_pedidos` lp
JOIN `{project}.{dataset}.productos` pr
  ON lp.id_producto = pr.id_producto
JOIN `{project}.{dataset}.categoria_productos` cp
  ON pr.id_categoria = cp.id_categoria
JOIN `{project}.{dataset}.pedidos` pe
  ON lp.id_pedido = pe.id_pedido
JOIN `{project}.{dataset}.clientes` c
  ON pe.id_cliente = c.id_cliente
LEFT JOIN `{project}.{dataset}.paises` pa
  ON c.pais = pa.id_pais
GROUP BY pa.nombre, cp.nombre
ORDER BY total_comprado DESC
"""

df7 = client.query(query).to_dataframe()
df7

c:\Users\sandy\Documents\GitHub\tc-sql-temuzon\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,pais,categoria,total_comprado
0,Guatemala,audio,205
1,Guatemala,wearables,189
2,Vietman,audio,181
3,Burkina Faso,wearables,180
4,Venezuela,wearables,179
5,Vietman,wearables,177
6,Estonia,wearables,175
7,Venezuela,audio,166
8,Burkina Faso,audio,161
9,Brunei Darussalam,audio,154
